# Issue #8 — Test Prediction-Based Roll Timing

Test whether predicted ZN liquidity crossover timing identifies a more cost-effective time to roll.

**This notebook uses only project data already created by earlier notebooks. It downloads nothing.**

Inputs:
- `zn_standardized_roll_panel.parquet` from Issue #14 / Step 7;
- `zn_early_indicator_panel.parquet` from `06_develop_early_indicators.ipynb`;
- `zn_crossover_events.parquet` from `05_define_crossover.ipynb`.

The early-indicator notebook already created the predictor variables, so they are not rebuilt here.

Because the project has no observed bid/ask quotes, transaction costs cannot be measured directly. We therefore compare roll schedules using the two cost proxies already derived from the existing daily calendar-spread OHLC:

1. `spread_roll_half_spread_cost_est` — a Roll-style implied half-spread estimate where identified;
2. `spread_half_range_cost_proxy` — half the daily calendar-spread high-low range as a deliberately conservative robustness proxy.

This distinction matters. The analysis tests whether the timing rule finds **cheaper-looking execution conditions in the available data**; it does not claim to recover true slippage for a specific order size.

**Issue #20 update:** timing snapshots now use the shared as-of implementation in `src/timing_model.py`. Earlier outputs were cleared because predictions and downstream cost comparisons must be recomputed. Relative-day units count upstream observations; exchange-session alignment and open-interest publication timing remain unverified. Use `python -m src.timing_model` for the model comparison and calendar audit.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(
    (p for p in (Path.cwd(), Path.cwd().parent)
     if (p / 'data' / 'processed' / 'zn_standardized_roll_panel.parquet').exists()),
    None
)
if project_root is None:
    raise FileNotFoundError(
        f"Could not locate standardized panel from {Path.cwd()}. "
        "Run 07_build_standardized_roll_panel.ipynb first."
    )

processed_dir = project_root / 'data' / 'processed'

standard = pd.read_parquet(processed_dir / 'zn_standardized_roll_panel.parquet')
early = pd.read_parquet(processed_dir / 'zn_early_indicator_panel.parquet')
events = pd.read_parquet(processed_dir / 'zn_crossover_events.parquet')

for df in (standard, early):
    df['date'] = pd.to_datetime(df['date'])
events['volume_crossover_date'] = pd.to_datetime(events['volume_crossover_date'])

required_standard = {
    'roll_id', 'date', 'bdays_to_fnd', 'next_volume_share', 'next_oi_share',
    'spread_volume', 'spread_volume_share',
    'spread_roll_half_spread_cost_est', 'spread_half_range_cost_proxy'
}
missing = required_standard - set(standard.columns)
if missing:
    raise ValueError(f"Standardized panel missing columns: {sorted(missing)}")

required_early = {
    'roll_id', 'date', 'bdays_to_fnd',
    'next_volume_share', 'next_volume_share_change_3d',
    'next_volume_share_change_5d', 'next_oi_share',
    'next_oi_share_change_3d', 'log_spread_volume'
}
missing = required_early - set(early.columns)
if missing:
    raise ValueError(f"Early-indicator panel missing columns: {sorted(missing)}")

print(f"Standardized panel: {standard['roll_id'].nunique()} rolls")
print(f"Early-indicator panel: {early['roll_id'].nunique()} rolls")
print(f"Crossover events: {len(events)} rolls")


## 1. Define an ex-ante prediction problem

To avoid using repeated roll-days as if they were independent observations, make one crossover-timing prediction per roll.

For each roll, use the indicator values available approximately **10 panel business days before FND**. The target is the observed volume-crossover day relative to FND.

Evaluation is **expanding-window out of sample**:
- each test roll is predicted using only earlier rolls;
- no later roll is allowed into its training sample;
- at least eight completed historical rolls are required before producing a prediction.

This is deliberately stricter than randomly splitting daily rows from the same rolls.

In [ ]:
DECISION_BDAY = -10
MIN_TRAIN_ROLLS = 8

FEATURES = [
    'next_volume_share',
    'next_volume_share_change_3d',
    'next_volume_share_change_5d',
    'next_oi_share',
    'next_oi_share_change_3d',
    'log_spread_volume',
]

# Issue #20: keep an actual as-of row; groupby.first() can fill values
# from other (including future) dates when the chosen row has missing OI.
import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.timing_model import prepare_decisions

decision = prepare_decisions(standard, DECISION_BDAY).rename(columns={
    'decision_date': 'date',
    'decision_bday': 'bdays_to_fnd',
    'actual_crossover_bday': 'volume_crossover_bdays_to_fnd',
    'actual_crossover_date': 'volume_crossover_date',
})
decision['volume_crossover_observed'] = decision['volume_crossover_bdays_to_fnd'].notna()

display(decision[[
    'roll_id', 'date', 'bdays_to_fnd',
    'volume_crossover_bdays_to_fnd'
] + FEATURES].head(12))


## 2. Predict crossover timing using only prior rolls

A small linear model is sufficient for the test here; the objective is not to maximize in-sample fit.

For every test roll:
- feature missing values are filled using **training-sample medians only**;
- features are standardized using **training-sample** means and standard deviations only;
- ordinary least squares is fitted on the prior rolls;
- the predicted crossover day is clipped to the interval from the decision day through FND.

We also construct a harder-to-beat simple benchmark: the median crossover day observed in prior rolls. If the feature model cannot beat that benchmark, the extra prediction machinery is not useful.

In [ ]:
prediction_rows = []

for i in range(len(decision)):
    test = decision.iloc[i]
    train = decision.iloc[:i].copy()

    train = train.loc[
        (train['roll_end'] < test['date'])
        & train['volume_crossover_observed'].fillna(False)
        & (train['volume_crossover_date'] > train['date'])
        & train['volume_crossover_bdays_to_fnd'].notna()
    ]

    if (pd.isna(test['volume_crossover_bdays_to_fnd'])
            or test['volume_crossover_date'] <= test['date']
            or len(train) < MIN_TRAIN_ROLLS):
        continue

    # Keep features that have enough historical observations and nonzero variation.
    active = []
    medians = {}
    means = {}
    stds = {}

    X_train_cols = []
    x_test_vals = []

    for feature in FEATURES:
        series = pd.to_numeric(train[feature], errors='coerce')
        if series.notna().sum() < max(5, MIN_TRAIN_ROLLS // 2):
            continue

        med = series.median()
        filled = series.fillna(med)
        sd = filled.std(ddof=0)
        if not np.isfinite(sd) or sd == 0:
            continue

        mu = filled.mean()

        active.append(feature)
        medians[feature] = med
        means[feature] = mu
        stds[feature] = sd
        X_train_cols.append(((filled - mu) / sd).to_numpy())

        test_val = pd.to_numeric(pd.Series([test[feature]]), errors='coerce').iloc[0]
        if pd.isna(test_val):
            test_val = med
        x_test_vals.append((test_val - mu) / sd)

    if len(active) == 0:
        continue

    X = np.column_stack(X_train_cols)
    X = np.column_stack([np.ones(len(X)), X])

    y = train['volume_crossover_bdays_to_fnd'].astype(float).to_numpy()

    # OLS via least squares; the small feature set is intentional.
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)

    x_test = np.array([1.0] + x_test_vals)
    raw_pred = float(x_test @ beta)

    # The strategy cannot choose the information date itself,
    # and this project is concerned with rolling no later than FND.
    predicted_bday = int(np.rint(np.clip(raw_pred, test['bdays_to_fnd'] + 1, 0)))

    historical_median = int(np.rint(
        train['volume_crossover_bdays_to_fnd'].median()
    ))

    prediction_rows.append({
        'roll_id': test['roll_id'],
        'decision_date': test['date'],
        'decision_bday': int(test['bdays_to_fnd']),
        'training_rolls': len(train),
        'features_used': ', '.join(active),
        'raw_predicted_crossover_bday': raw_pred,
        'predicted_crossover_bday': predicted_bday,
        'historical_median_crossover_bday': historical_median,
        'actual_crossover_bday': test['volume_crossover_bdays_to_fnd'],
        'actual_crossover_date': test['volume_crossover_date'],
    })

predictions = pd.DataFrame(prediction_rows)

if predictions.empty:
    raise ValueError("No out-of-sample predictions were produced.")

predictions['prediction_error_bdays'] = (
    predictions['predicted_crossover_bday']
    - predictions['actual_crossover_bday']
)
predictions['historical_median_error_bdays'] = (
    predictions['historical_median_crossover_bday']
    - predictions['actual_crossover_bday']
)

display(predictions)

print("Prediction MAE, panel-day units:",
      predictions['prediction_error_bdays'].abs().mean())
print("Historical-median MAE, panel-day units:",
      predictions['historical_median_error_bdays'].abs().mean())


## Prediction Results

Compare the computed errors above. Issue #20 corrected snapshot selection so missing decision-day values cannot be filled from another date, and forecasts cannot select the same day's execution. Earlier saved results (OLS MAE 1.00 versus historical-median MAE 0.26) used the old snapshot rule and are reproduced separately by `python -m src.timing_model`.

The corrected OLS still underperforms the historical median on the available sample. The Issue #20 module compares regularized alternatives using nested chronological validation; see `docs/issue-20-model-improvements.md` for the measured results and date-alignment limitations.


## 3. Map each timing rule to an actual available roll-day observation

Compare five implementable schedules:

- **prediction_based** — the model's predicted crossover day;
- **historical_median** — median crossover timing from prior rolls only;
- **fixed_-5** — five business days before FND;
- **fixed_-3** — three business days before FND;
- **fixed_-1** — one business day before FND.

`hindsight_actual_crossover` is included only as a diagnostic. It is not an implementable benchmark because it uses the realized event itself.

In [ ]:
def nearest_roll_row(grp, target_bday):
    grp = grp.copy()
    grp['distance_to_target'] = (grp['bdays_to_fnd'] - target_bday).abs()
    # If equally close, choose the earlier date to avoid silently looking forward.
    return grp.sort_values(['distance_to_target', 'date']).iloc[0]

eval_rows = []

for _, pred in predictions.iterrows():
    grp = standard.loc[standard['roll_id'] == pred['roll_id']].sort_values('date')
    if grp.empty:
        continue

    schedules = {
        'prediction_based': pred['predicted_crossover_bday'],
        'historical_median': pred['historical_median_crossover_bday'],
        'fixed_-5': -5,
        'fixed_-3': -3,
        'fixed_-1': -1,
        'hindsight_actual_crossover': pred['actual_crossover_bday'],
    }

    for strategy, target_bday in schedules.items():
        if pd.isna(target_bday):
            continue

        row = nearest_roll_row(grp, float(target_bday))
        eval_rows.append({
            'roll_id': pred['roll_id'],
            'strategy': strategy,
            'target_bday': float(target_bday),
            'selected_bday': row['bdays_to_fnd'],
            'selected_date': row['date'],
            'spread_roll_half_spread_cost_est': row['spread_roll_half_spread_cost_est'],
            'spread_half_range_cost_proxy': row['spread_half_range_cost_proxy'],
            'spread_volume': row['spread_volume'],
            'spread_volume_share': row['spread_volume_share'],
            'next_volume_share': row['next_volume_share'],
            'next_oi_share': row['next_oi_share'],
        })

evaluation = pd.DataFrame(eval_rows)
display(evaluation.head(18))


## 4. Compare transaction-cost proxies

Do not judge the strategy only by average cost. *Dynamic Hedging* emphasizes that liquidity losses and forced execution can be nonlinear and badly represented by a single central estimate.

Accordingly, report:
- coverage;
- median and mean;
- 90th percentile;
- worst observed cost;
- spread volume at the chosen date.

A timing rule that slightly improves the mean but creates a worse tail is not obviously better.

In [ ]:
def q90(x):
    return x.quantile(0.90)

summary = (
    evaluation.groupby('strategy')
    .agg(
        rolls=('roll_id', 'nunique'),
        roll_spread_estimate_observations=('spread_roll_half_spread_cost_est', 'count'),
        roll_spread_cost_median=('spread_roll_half_spread_cost_est', 'median'),
        roll_spread_cost_mean=('spread_roll_half_spread_cost_est', 'mean'),
        roll_spread_cost_p90=('spread_roll_half_spread_cost_est', q90),
        roll_spread_cost_worst=('spread_roll_half_spread_cost_est', 'max'),
        range_proxy_observations=('spread_half_range_cost_proxy', 'count'),
        range_proxy_median=('spread_half_range_cost_proxy', 'median'),
        range_proxy_mean=('spread_half_range_cost_proxy', 'mean'),
        range_proxy_p90=('spread_half_range_cost_proxy', q90),
        range_proxy_worst=('spread_half_range_cost_proxy', 'max'),
        median_spread_volume=('spread_volume', 'median'),
        median_spread_volume_share=('spread_volume_share', 'median'),
        median_next_volume_share=('next_volume_share', 'median'),
    )
    .sort_index()
)

display(summary)


## 5. Paired comparison against each benchmark

The cleanest comparison is within the same roll. For each benchmark, subtract its cost from the prediction-based cost.

- negative difference = prediction-based timing was cheaper;
- positive difference = benchmark was cheaper.

We do this separately for the Roll-style half-spread estimate and the high-low range proxy.

In [ ]:
BENCHMARKS = ['historical_median', 'fixed_-5', 'fixed_-3', 'fixed_-1']

def paired_comparison(metric):
    pred = (
        evaluation.loc[evaluation['strategy'] == 'prediction_based',
                       ['roll_id', metric]]
        .rename(columns={metric: 'predicted_cost'})
    )

    rows = []
    for benchmark in BENCHMARKS:
        bench = (
            evaluation.loc[evaluation['strategy'] == benchmark,
                           ['roll_id', metric]]
            .rename(columns={metric: 'benchmark_cost'})
        )
        paired = pred.merge(bench, on='roll_id', how='inner').dropna()
        paired['difference'] = paired['predicted_cost'] - paired['benchmark_cost']

        rows.append({
            'metric': metric,
            'benchmark': benchmark,
            'paired_rolls': len(paired),
            'median_pred_minus_benchmark': paired['difference'].median(),
            'mean_pred_minus_benchmark': paired['difference'].mean(),
            'prediction_cheaper_fraction': (
                (paired['difference'] < 0).mean() if len(paired) else np.nan
            ),
            'prediction_not_worse_fraction': (
                (paired['difference'] <= 0).mean() if len(paired) else np.nan
            ),
            'worst_pred_minus_benchmark': (
                paired['difference'].max() if len(paired) else np.nan
            ),
        })

    return pd.DataFrame(rows)

paired_roll = paired_comparison('spread_roll_half_spread_cost_est')
paired_range = paired_comparison('spread_half_range_cost_proxy')

paired_results = pd.concat([paired_roll, paired_range], ignore_index=True)
display(paired_results)


## 6. Inspect timing accuracy and execution implications

The timing model is useful only if the execution results survive comparison with simple schedules.

Interpretation should therefore separate two questions:

1. **Prediction:** Does the early-indicator model forecast crossover timing more accurately than the historical-median rule?
2. **Execution:** On the dates it selects, are the available transaction-cost proxies lower without materially worse tail outcomes?

If the answer to (1) is yes but (2) is no, better crossover prediction did not translate into a better roll.
If the cost result exists only for one proxy or only in the mean, the evidence is weak.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    predictions['actual_crossover_bday'],
    predictions['predicted_crossover_bday']
)
lo = min(
    predictions['actual_crossover_bday'].min(),
    predictions['predicted_crossover_bday'].min()
)
hi = max(
    predictions['actual_crossover_bday'].max(),
    predictions['predicted_crossover_bday'].max()
)
ax.plot([lo, hi], [lo, hi], linestyle='--')
ax.set(
    xlabel='Actual crossover business day to FND',
    ylabel='Predicted crossover business day to FND',
    title='Out-of-sample predicted vs. actual crossover timing'
)
ax.grid(alpha=0.25)
plt.show()

display(predictions[[
    'roll_id', 'training_rolls',
    'predicted_crossover_bday', 'actual_crossover_bday',
    'prediction_error_bdays',
    'historical_median_crossover_bday',
    'historical_median_error_bdays'
]])


## Conclusion

Reassess the execution comparison using the freshly computed tables above. The original saved run found mixed results across cost proxies; those tables must be regenerated after correcting timing snapshots in Issue #20.

The crossover model comparison is documented in `docs/issue-20-model-improvements.md`. The historical-median timing rule remains preferred on the available sample. Improved crossover accuracy alone would not establish lower execution costs, and neither daily-price proxy is observed slippage or an observed bid-ask spread.


In [ ]:
pred_path = processed_dir / 'zn_roll_timing_predictions.parquet'
eval_path = processed_dir / 'zn_roll_timing_cost_comparison.parquet'
summary_path = processed_dir / 'zn_roll_timing_cost_summary.parquet'

predictions.to_parquet(pred_path, index=False)
evaluation.to_parquet(eval_path, index=False)
summary.reset_index().to_parquet(summary_path, index=False)
